<a href="https://colab.research.google.com/github/AHMADAMJAD0/prof_amadeo_task/blob/main/MyHill_nerode_agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
#!/usr/bin/env python3
"""Teaching agent for Myhill-Nerode. Requires Python 3.9 or later.

Enumerates suffixes and records the first distinguishing witness for each pair.
Use --exact to resolve pending pairs by exhaustive finite-product search.
The candidate DFA is always checked exactly. Use --json to export the trace.

Examples:
    python myhill_nerode_agent.py
    python myhill_nerode_agent.py --limit 0 --exact
    python myhill_nerode_agent.py --prefixes "" 0 101 1 10 --exact --json

AI orchestration:
    Run with --limit 1 --json first. Inspect unresolved pairs before using
    --exact. Explain the evidence, stopping condition and lower bound.
    Manually check one witness. Keep this source file intact.
"""

import argparse
from collections import deque
from itertools import combinations, product
import json


ALPHABET = "01"
CANDIDATE = {
    "A": {"0": "A", "1": "C"},
    "B": {"0": "A", "1": "C"},
    "C": {"0": "D", "1": "A"},
    "D": {"0": "D", "1": "B"},
}


def belongs(word):
    """Direct membership oracle, independent of the transition table."""
    return word.endswith("01") and word.count("1") % 2 == 0


def reference_step(state, symbol):
    """Reference DFA tracking parity and the last two symbols."""
    parity, ending = state
    return ((parity + int(symbol)) % 2, (ending + symbol)[-2:])


def reference_accepts(state):
    return state == (0, "01")


def reference_state(word):
    return (word.count("1") % 2, word[-2:])


def difference(initial, advance, disagree):
    """BFS on a finite product. Returns (witness or None, visited pairs)."""
    queue = deque([(initial, "")])
    seen = {initial}
    visited = 0
    while queue:
        pair, suffix = queue.popleft()
        visited += 1
        if disagree(pair):
            return suffix, visited
        for symbol in ALPHABET:
            next_pair = advance(pair, symbol)
            if next_pair not in seen:
                seen.add(next_pair)
                queue.append((next_pair, suffix + symbol))
    return None, visited


def record(row, suffix, method):
    x, y = row["prefixes"]
    words = [x + suffix, y + suffix]
    results = [belongs(word) for word in words]
    if results[0] == results[1]:
        raise RuntimeError("The membership oracle rejected the computed witness.")
    row.update(state="distinguishable", suffix=suffix,
                concatenations=words, accepted_results=results, method=method)


def analyze(prefixes, limit, exact):
    rows = [{"prefixes": [x, y], "state": "no witness within the bound"}
             for x, y in combinations(prefixes, 2)]
    pending = rows[:]
    examined = 0
    for length in range(limit + 1):
        for letters in product(ALPHABET, repeat=length):
            if not pending:
                break
            suffix = "".join(letters)
            examined += 1
            remaining = []
            for row in pending:
                x, y = row["prefixes"]
                if belongs(x + suffix) != belongs(y + suffix):
                    record(row, suffix, "enumeration by length")
                else:
                    remaining.append(row)
            pending = remaining
        if not pending:
            break

    stopping_condition = "all pairs resolved" if not pending else "bound reached"
    if pending and exact:
        stopping_condition = "pending pairs resolved by exact search"
        for row in pending:
            initial = tuple(map(reference_state, row["prefixes"]))
            suffix, visited = difference(
                initial,
                lambda pair, s: tuple(reference_step(state, s) for state in pair),
                lambda pair: reference_accepts(pair[0]) != reference_accepts(pair[1]),
            )
            row["visited_pairs"] = visited
            if suffix is None:
                row.update(state="equivalent", method="exhaustive product search")
            else:
                record(row, suffix, "exhaustive product search")

    counterexample, visited = difference(
        ("A", (0, "")),
        lambda pair, s: (CANDIDATE[pair[0]][s], reference_step(pair[1], s)),
        lambda pair: (pair[0] == "B") != reference_accepts(pair[1]),
    )
    # The lower bound uses a pairwise distinguishable set, even when
    # additional requested prefixes are equivalent to each other.
    witnesses = {tuple(row["prefixes"]) for row in rows
                if row["state"] == "distinguishable"}
    group = []
    for prefix in prefixes:
        if all((previous, prefix) in witnesses for previous in group):
            group.append(prefix)

    return {
        "limit": limit, "suffixes_examined": examined, "stopping_condition": stopping_condition,
        "pairs": rows, "lower_bound_prefixes": group, "lower_bound": len(group),
        "dfa": {
            "initial_state": "A", "accepting_states": ["B"], "transitions": CANDIDATE,
            "equivalent_to_L": counterexample is None,
            "counterexample": counterexample, "visited_pairs": visited,
            "minimality_certified": counterexample is None and len(group) == len(CANDIDATE),
        },
    }


def display(result):
    def display_word(word):
        return word or "ε"

    print("L = binary strings ending in 01 with an even number of ones.")
    for row in result["pairs"]:
        x, y = map(display_word, row["prefixes"])
        print(f"{x} and {y} → {row['state']}", end="")
        if "suffix" in row:
            evidence = ", ".join(
                f"{display_word(word)} {'∈' if accepted else '∉'} L"
                for word, accepted in zip(row["concatenations"], row["accepted_results"])
            )
            print(f" with z = {display_word(row['suffix'])} ({evidence})", end="")
        print()
    print(f"Stopping condition: {result['stopping_condition']}.")
    print(f"Certified lower bound: {result['lower_bound']} states.")
    dfa = result["dfa"]
    if dfa["equivalent_to_L"]:
        print("The candidate DFA recognizes L. The reachable product was exhausted.")
    else:
        print(f"Counterexample to the candidate DFA: {display_word(dfa['counterexample'])}.")
    if dfa["minimality_certified"]:
        print("Minimality certified: 4 states.")


def main():
    parser = argparse.ArgumentParser(description=__doc__,
                                     formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("--limit", type=int, default=2,
                        help="maximum suffix length for initial enumeration (0 to 12)")
    parser.add_argument("--prefixes", nargs="+", default=["ε", "101", "1", "10"],
                        help="distinct binary prefixes; use ε or an empty argument for the empty string")
    parser.add_argument("--exact", action="store_true",
                        help="resolve pending pairs using a finite product")
    parser.add_argument("--json", action="store_true", help="emit the trace as JSON")
    args, _ = parser.parse_known_args()  # parse_known_args ignores Jupyter/Colab kernel flags
    prefixes = ["" if word == "ε" else word for word in args.prefixes]
    if not 0 <= args.limit <= 12:
        parser.error("--limit must be between 0 and 12")
    if not 2 <= len(prefixes) <= 16 or len(set(prefixes)) != len(prefixes):
        parser.error("provide between 2 and 16 distinct prefixes")
    if any(set(word) - set(ALPHABET) for word in prefixes):
        parser.error("use only 0, 1 or ε")
    result = analyze(prefixes, args.limit, args.exact)
    if args.json:
        print(json.dumps(result, ensure_ascii=False, indent=2))
    else:
        display(result)


if __name__ == "__main__":
    main()

L = binary strings ending in 01 with an even number of ones.
ε and 101 → distinguishable with z = ε (ε ∉ L, 101 ∈ L)
ε and 1 → distinguishable with z = 01 (01 ∉ L, 101 ∈ L)
ε and 10 → distinguishable with z = 1 (1 ∉ L, 101 ∈ L)
101 and 1 → distinguishable with z = ε (101 ∈ L, 1 ∉ L)
101 and 10 → distinguishable with z = ε (101 ∈ L, 10 ∉ L)
1 and 10 → distinguishable with z = 1 (11 ∉ L, 101 ∈ L)
Stopping condition: all pairs resolved.
Certified lower bound: 4 states.
The candidate DFA recognizes L. The reachable product was exhausted.
Minimality certified: 4 states.


In [6]:
import json, subprocess

with open("/content/bounded_trace.json") as f:
    trace = json.load(f)

# Extract unique prefixes in order
seen = []
for pair in trace["pairs"]:
    for p in pair["prefixes"]:
        if p not in seen:
            seen.append(p)

prefixes = seen
print("Prefixes found:", prefixes)  # check what was extracted

# Build args — empty string becomes ε
prefix_args = [p if p != "" else "ε" for p in prefixes]
print("Prefix args:", prefix_args)  # check conversion

cmd = (
    ["python", "myhill_nerode_agent.py",
     "--limit", str(trace["limit"]),
     "--prefixes"] + prefix_args + ["--exact", "--json"]
)
print("Command:", cmd)  # check full command

result = subprocess.run(cmd, capture_output=True, text=True)
print("STDERR:", result.stderr)  # show any errors

new_trace = json.loads(result.stdout)
print(json.dumps(new_trace, indent=2))

Prefixes found: ['', '0', '101', '1', '10']
Prefix args: ['ε', '0', '101', '1', '10']
Command: ['python', 'myhill_nerode_agent.py', '--limit', '1', '--prefixes', 'ε', '0', '101', '1', '10', '--exact', '--json']
STDERR: python3: can't open file '/content/myhill_nerode_agent.py': [Errno 2] No such file or directory



JSONDecodeError: Expecting value: line 1 column 1 (char 0)